# 07 — Memory Patterns

Three memory strategies: full buffer, sliding window, and summary memory.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

## Pattern 1: Full Buffer Memory

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])
chain = prompt | llm | StrOutputParser()

history = []
for msg in ["My name is Alice and I'm a data scientist.", "I work at Acme Corp on recommendation systems.", "What do you know about me so far?"]:
    print(f"User: {msg}")
    response = chain.invoke({"input": msg, "history": history})
    print(f"AI: {response}")
    history.append(HumanMessage(content=msg))
    history.append(AIMessage(content=response))
print(f"[Messages stored: {len(history)}]")

## Pattern 2: Sliding Window Memory

In [ ]:
window_size = 4  # 2 exchanges
history = []
for msg in ["My favorite color is blue.", "My favorite food is sushi.", "My favorite movie is Inception.", "What is my favorite color?", "What is my favorite movie?"]:
    windowed = history[-window_size:]
    print(f"User: {msg}")
    response = chain.invoke({"input": msg, "history": windowed})
    print(f"AI: {response}")
    history.append(HumanMessage(content=msg))
    history.append(AIMessage(content=response))

## Pattern 3: Summary Memory

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise.\n\nSummary of earlier conversation:\n{summary}"),
    MessagesPlaceholder("recent_history"),
    ("human", "{input}"),
])
summary_chain = summary_prompt | llm | StrOutputParser()

summary = "No prior conversation."
recent_history = []

for msg in ["I'm planning a trip to Japan in April.", "I want to visit Tokyo, Kyoto, and Osaka.", "My budget is around $3000 for two weeks.", "I love Japanese food, especially ramen.", "Can you summarise my trip plans so far?"]:
    if len(recent_history) >= 4:
        summarize_chain = ChatPromptTemplate.from_template(
            "Summarise this conversation in 2-3 sentences:\n\nPrevious summary: {old_summary}\n\nRecent messages:\n{messages}"
        ) | llm | StrOutputParser()
        messages_text = "\n".join(f"{'User' if isinstance(m, HumanMessage) else 'AI'}: {m.content}" for m in recent_history)
        summary = summarize_chain.invoke({"old_summary": summary, "messages": messages_text})
        recent_history = []
        print(f"[Summarised: {summary[:80]}...]")
    print(f"User: {msg}")
    response = summary_chain.invoke({"input": msg, "summary": summary, "recent_history": recent_history})
    print(f"AI: {response}")
    recent_history.append(HumanMessage(content=msg))
    recent_history.append(AIMessage(content=response))